# Day 1 -- From pixels to vision transformers

Welcome. Over the next three hours you will build **five image classifiers** on the same
medical images and watch each one beat the last:

> **logistic regression -> MLP -> CNN -> ResNet -> Vision Transformer**

Each rung up the ladder adds one idea, and you will *see* the accuracy climb because of it.
That climb is the whole story of how computer vision got good.

### The dataset and the task
We use **APTOS-2019**: color photographs of the back of the eye (the *retina*, or *fundus*).
Diabetes damages the tiny blood vessels there, a disease called **diabetic retinopathy (DR)**.
Our task is the one actually deployed in clinics: **referable DR** -- does this eye show
enough disease that the person should see a specialist? **Yes or no.** A binary screening call.

### What you'll be able to do by the end
- Explain what an image *is* to a model, and why pixels alone are hard.
- Say what each model adds: non-linearity, spatial structure, pretraining, attention.
- Read a **learning curve** and a **confusion matrix**, and know why accuracy can mislead.
- Understand *transfer learning* -- the single most useful trick in applied vision.

### How the lab works
Fill in the `# TODO` lines. Stuck? Ask Claude, then make sure you understand the answer
before moving on. The point isn't to finish fastest, it's to understand *why each rung
beats the one below it.*

In [ ]:
# Setup: on Colab, grab the course files. Locally (already in the repo) this is a no-op.
import os, sys
if not os.path.exists("common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day1_ladder")
sys.path.insert(0, ".")
sys.path.insert(0, "../_shared")
import colab_setup
colab_setup.ensure()
colab_setup.gpu_check()

In [ ]:
import os, sys
import torch
import numpy as np
import common
# nbfig lives in notebooks/_shared; add it relative to common.py so this cell
# works even if the setup cell above wasn't run, and from any working directory.
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(common.__file__)), "..", "_shared"))
import nbfig            # Colab-safe branded plotting (matches the slide figures)
nbfig.use()

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
results = {}   # we'll collect each model's validation accuracy here

## 0. The data: pixels, eyes, and a yes/no question

Before any modeling, spend real time looking at the data. Good practitioners always do.

### 0.1 The clinical task
About **1 in 3** people with diabetes develop some retinopathy, and it is a leading cause
of blindness in working-age adults, yet it is **preventable** if caught early. The catch:
catching it needs a trained eye specialist, and there are far too few. That mismatch -- huge
routine need, scarce expertise -- is exactly where AI screening helps. Let's look at the eyes.

In [ ]:
train_loader, val_loader = common.get_loaders(size=224, batch_size=32)
images, labels = next(iter(train_loader))
print("one batch:", images.shape, "  <- (batch, channels, height, width)")

# Look at eight real fundus images with their referable / not-referable labels.
fig, axes = nbfig.fig(2, 4, figsize=(11, 5.6))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(common._denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(common.GRADE_NAMES[int(lab)], fontsize=11,
                 color=(nbfig.DEEPPINK if int(lab) else nbfig.INK))
    ax.axis("off")
nbfig.show(fig, "Eight eyes: can you tell which need a doctor?")

### 0.2 An image is just numbers
To a model, this photo is not a picture -- it's a grid of numbers. A *color* image is
**three** grids stacked: how much red, green, and blue at each pixel. Everything today
operates on those numbers.

In [ ]:
common.show_rgb_split(images[0])      # the same eye as red / green / blue grids
common.show_pixel_histogram(images[0])  # the distribution of pixel brightnesses

### 0.3 Normalization: put every channel on the same scale
Networks train best when their inputs are **centered at zero with a similar spread**.
**Standardization** does exactly that: for each color channel, subtract that channel's mean and
divide by its standard deviation. The result has mean ~0 and std ~1. Watch the per-channel means
collapse onto the zero line below.

Heads-up: the middle panel (the standardized image) looks washed-out and bluish, **not** like a
real photo -- that's expected. A retina is almost all red/orange with barely any blue, so
stretching every channel to the same spread blows the faint blue channel way up. There's no
"natural" way to display a standardized tensor; the model doesn't need one, it just reads the
numbers.

In [ ]:
# Standardize each channel using THIS data's own mean and std -- the textbook recipe.
raw_batch = torch.stack([common._denorm(im) for im in images])  # N,3,H,W back in 0..1
ch_mean = raw_batch.mean(dim=(0, 2, 3))        # per-channel mean
ch_std = raw_batch.std(dim=(0, 2, 3))          # per-channel std
standardized = (raw_batch - ch_mean[:, None, None]) / ch_std[:, None, None]

raw0, std0 = raw_batch[0], standardized[0]
disp = (std0 - std0.min()) / (std0.max() - std0.min())  # rescale just to view

fig, (a1, a2, a3) = nbfig.fig(1, 3, figsize=(11, 3.6))
a1.imshow(raw0.permute(1, 2, 0).numpy()); a1.set_title("raw (0..1)", fontsize=11); a1.axis("off")
a2.imshow(disp.permute(1, 2, 0).numpy()); a2.set_title("standardized (rescaled to view)", fontsize=11); a2.axis("off")
x = np.arange(3)
a3.bar(x - 0.2, raw_batch.mean((0, 2, 3)).numpy(), 0.4, color=nbfig.TURQUOISE, label="raw mean")
a3.bar(x + 0.2, standardized.mean((0, 2, 3)).numpy(), 0.4, color=nbfig.DEEPPINK, label="standardized mean")
a3.axhline(0, color=nbfig.MUTED, lw=0.8); a3.set_xticks(x); a3.set_xticklabels(["R", "G", "B"])
a3.set_title("channel means: ~0 after", fontsize=11); a3.legend()
nbfig.show(fig, "Standardization: subtract the mean, divide by the std -> centered at zero")
print("standardized channel means:", [round(v, 3) for v in standardized.mean((0, 2, 3)).tolist()])

> **One practical footnote.** Our actual training pipeline standardizes with *fixed* **ImageNet**
> constants (mean `0.485/0.456/0.406`, std `0.229/0.224/0.225`) rather than each batch's own
> stats. Two reasons: the transform must be identical at train and test time (per-batch stats
> would drift), and the pretrained ResNet/ViT we load later were trained with exactly those
> numbers. It's the same idea -- a fixed, shared scale -- it just leaves our darker-than-average
> retinas sitting a little below zero instead of exactly on it.

### 0.4 Know your labels (this matters more than you think)
Before trusting any accuracy number, check how many examples of each class you have. If
most eyes are "not referable," a lazy model can score high by always guessing the majority.
Hold that thought -- we'll catch a model doing exactly that at the end.

In [ ]:
import numpy as np
# count classes across the validation set
all_labels = np.concatenate([y.numpy() for _, y in val_loader])
counts = [int((all_labels == c).sum()) for c in range(common.NUM_CLASSES)]

fig, ax = nbfig.fig(figsize=(5.5, 3.2))
bars = ax.bar(common.GRADE_NAMES, counts, color=[nbfig.TURQUOISE, nbfig.DEEPPINK])
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width() / 2, c, str(c), ha="center", va="bottom",
            fontweight="bold", family="DejaVu Sans Mono")
ax.set_ylabel("validation images")
nbfig.show(fig, "Class balance")
print("majority-class baseline accuracy:", f"{max(counts) / sum(counts):.3f}")

### 0.5 Augmentation: turn one eye into many
We have only a few thousand images. **Augmentation** makes small random changes -- flip,
rotate -- every time an image is used, so the model sees more variety and can't just
memorize specific photos. Here is the *same eye*, fifteen times, run through the real
training pipeline. Each one is a slightly different training example, for free.

In [ ]:
import torchvision.transforms as T
pil = T.ToPILImage()(common._denorm(images[0]))
aug = common._transform(224, train=True)   # the actual training-time augmentation

fig, axes = nbfig.fig(3, 5, figsize=(11, 7))
for ax in axes.ravel():
    shown = common._denorm(aug(pil))        # apply augmentation, then de-normalize to view
    ax.imshow(shown.permute(1, 2, 0).numpy()); ax.axis("off")
nbfig.show(fig, "The same eye, 15 random augmentations")

### 0.6 Augmentations we deliberately *avoid* (and why)
Augmentation is not a free-for-all. In everyday-photo tasks (cats, cars) people throw on strong
**color jitter, shearing, perspective warps, vertical flips** -- a cat is still a cat upside-down
and tinted blue. **Medicine is different: some "harmless" augmentations destroy the signal.** The
exact things a clinician reads off a retina -- the *color* of a hemorrhage, the *shape* of the
vessels, the *position* of a lesion relative to the macula -- are the things these transforms
mangle. Below we crank each one up so you can *see* the damage, then we explain why it's off the table.

In [ ]:
# Crank each "risky" augmentation way up so the damage is obvious.
risky = {
    "original":        pil,
    "color jitter":    T.ColorJitter(brightness=0.0, contrast=0.0, saturation=0.0, hue=0.45)(pil),
    "heavy shear":     T.functional.affine(pil, angle=0, translate=(0, 0), scale=1.0, shear=[35, 20]),
    "perspective warp": T.RandomPerspective(distortion_scale=0.6, p=1.0)(pil),
    "vertical flip":   T.functional.vflip(pil),
    "huge zoom-crop":  T.RandomResizedCrop(224, scale=(0.15, 0.15))(pil),
}
fig, axes = nbfig.fig(2, 3, figsize=(11, 7.2))
for ax, (name, im) in zip(axes.ravel(), risky.items()):
    ax.imshow(im); ax.set_title(name, fontsize=12,
              color=(nbfig.INK if name == "original" else nbfig.DEEPPINK)); ax.axis("off")
nbfig.show(fig, "Too far: augmentations that corrupt the diagnosis")

**Why each one is dangerous here:**
- **Color jitter / hue shift** -- diabetic retinopathy is graded partly on the *color* of
  lesions (bright yellow exudates, deep-red hemorrhages). Recolor the image and you can turn a
  textbook finding into something that looks like a different disease.
- **Shear / perspective warp** -- these bend straight anatomy. Vessel calibre and the shape of
  microaneurysms are diagnostic; warping invents geometry the eye never had.
- **Vertical flip** -- a retina has a real top and bottom. DR is graded by *where* lesions sit
  relative to the macula and optic disc; flipping vertically teaches the model anatomy that
  doesn't exist in any real patient. (A horizontal flip is the borderline-OK exception -- a left
  eye is roughly a mirror of a right eye, so we *do* allow that one.)
- **Huge zoom-crop** -- crop too aggressively and the actual lesion can fall outside the frame,
  so the image no longer supports its own label.

The rule of thumb: **augment only in ways a real patient or camera could plausibly produce.**
Small rotations and a horizontal flip pass that test; the transforms above do not -- which is
exactly why our training pipeline (Section 0.5) sticks to the gentle ones.

### 0.7 Play with it: your own augmentation dials
Now grab the controls. Drag the sliders below and watch a real eye transform in real time, no
training, instant feedback. Try to answer for yourself: *which* of these still looks like a valid
patient photo, and which ones cross the line we just talked about? (The sliders need Colab or
Jupyter widgets; if you don't see them, run the previous cells first.)

In [ ]:
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

# self-heal: works even if you jumped straight here after a kernel restart
try:
    common, device, tr224
except NameError:
    import sys; sys.path.insert(0, ".")
    import common
    _nb, device, tr224, _va = common.playground_setup()

# one clean sample eye to experiment on
_imgs, _ = next(iter(tr224))
BASE = TF.to_pil_image(common._denorm(_imgs[0]))

def show_augment(rotate=0, brightness=1.0, contrast=1.0, blur=0, zoom=1.0, hflip=False):
    img = TF.adjust_contrast(TF.adjust_brightness(BASE, brightness), contrast)
    if zoom > 1.0:
        w, h = img.size
        img = TF.resize(TF.center_crop(img, [int(h / zoom), int(w / zoom)]), [h, w])
    img = TF.rotate(img, rotate)
    if hflip:
        img = TF.hflip(img)
    if blur > 0:
        img = TF.gaussian_blur(img, kernel_size=2 * blur + 1)
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"rotate {rotate}deg | bright {brightness:.1f} | contrast {contrast:.1f} | "
                 f"blur {blur} | zoom {zoom:.1f} | hflip {hflip}", fontsize=8)
    plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
    interact(show_augment,
             rotate=IntSlider(value=0, min=-45, max=45, step=5),
             brightness=FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1),
             contrast=FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1),
             blur=IntSlider(value=0, min=0, max=8, step=1),
             zoom=FloatSlider(value=1.0, min=1.0, max=2.5, step=0.1),
             hflip=Checkbox(value=False))
except ImportError:
    print("ipywidgets not available -- showing a few fixed examples instead.")
    for kw in [dict(), dict(rotate=25), dict(brightness=1.6), dict(blur=5), dict(zoom=1.8, hflip=True)]:
        show_augment(**kw)

## 1. Logistic regression -- the simplest classifier

We start at the bottom rung. Flatten each image into one long row of numbers and fit a
**linear** model: it draws a single straight boundary between "referable" and "not." We
use small 64x64 images so it runs in seconds.

**Predict first:** a coin flip gets ~50%, and always-guessing-the-majority gets the
number you just printed. How much better can a straight line on raw pixels do? Write a guess.

In [ ]:
from sklearn.linear_model import LogisticRegression

tr64, va64 = common.get_loaders(size=64, batch_size=64)
Xtr, ytr = common.flatten_for_classical(tr64)
Xva, yva = common.flatten_for_classical(va64)
print("flattened features per image:", Xtr.shape[1])

# TODO: make a LogisticRegression with max_iter=1000
# TODO: fit the classifier on the training features Xtr, ytr

preds = clf.predict(Xva)
acc = (preds == yva).mean()
results["logreg"] = acc
print(f"logistic regression val accuracy: {acc:.3f}")

### 1.1 Look at *how* it's right and wrong
Accuracy is one number; a **confusion matrix** shows the mistakes. Rows are the truth,
columns are the guess. The diagonal is correct; off-diagonal is where it slips.

In [ ]:
nbfig.confusion(yva, preds, common.GRADE_NAMES, text="Logistic regression").show()

Logistic regression treats every pixel as independent and can only draw straight-line
boundaries. It has no idea that neighboring pixels form vessels, spots, or shapes. Hold
that thought -- it's the ceiling every later model breaks through.

## 2. Multi-layer perceptron -- stacking non-linear layers

Same flattened pixels, but now we stack layers with non-linearities between them, so the
model can **bend** its decision boundary instead of using one straight cut. It still has
no notion of space, though -- shuffle the pixels and it wouldn't notice.

In [ ]:
# TODO: build an MLP with common.make_mlp; in_features is 3*64*64 (the flattened size)
# TODO: train it: common.train_model(mlp, tr64, va64, epochs=15, lr=1e-3, device=device)
results["mlp"] = history[-1][1]
print(f"mlp val accuracy: {history[-1][1]:.3f}")

### 2.1 The learning curve
Unlike logreg, a neural net learns *gradually*, one epoch (one pass through the data) at a
time. The curve below shows validation accuracy climbing while the training loss falls --
that's learning happening. A flat curve means it's stuck; a falling-then-rising-loss curve
would mean trouble.

In [ ]:
nbfig.learning_curve(history, text="MLP: accuracy up, loss down").show()

*Did it beat logreg by much?* Usually only a little. Extra layers help the model bend, but
it still can't see that pixels next to each other belong together. That missing idea --
**spatial structure** -- is the next rung, and it's a big one.

## 3. Convolutional neural network -- seeing in 2D

A **CNN** slides small filters across the image, so it understands *spatial* structure:
edges, blobs, textures, and where they are. Each filter is a little pattern-detector that
fires when it sees its pattern anywhere in the image. Now we switch to the full 224x224 images.

**Predict:** how much will accuracy jump versus the MLP?

In [ ]:
tr224, va224 = common.get_loaders(size=224, batch_size=32)

# TODO: build the CNN with common.make_small_cnn()
# TODO: train it for 15 epochs at lr=1e-3 on the 224px loaders
results["cnn"] = history[-1][1]
print(f"cnn val accuracy: {history[-1][1]:.3f}")

In [ ]:
nbfig.learning_curve(history, text="CNN: a steeper climb").show()

### 3.1 What did it learn to see?
We can peek at the filters in the very first convolutional layer. Nobody told the network
what to look for -- it *learned* these little edge and color detectors on its own, just
from trying to predict the label.

In [ ]:
common.show_first_layer_filters(cnn)

### 3.2 Its mistakes

In [ ]:
res = common.evaluate_classifier(lambda b: cnn(b).argmax(1), va224, device)
nbfig.confusion(res["y"], res["pred"], common.GRADE_NAMES, text="CNN confusion matrix").show()

That jump over the MLP is convolutions earning their keep: the model can finally use the
*arrangement* of pixels, not just their values. But we trained it from scratch on a few
thousand images. The next rung borrows a head start from millions.

### 3.3 Your turn: the knobs that change everything

A model isn't a fixed thing, it's a machine with **dials**. The ones you'll touch most:

- **Learning rate** -- the *step size* the model takes downhill while learning (remember the ball
  rolling down a hill). Too small and it crawls; too big and it overshoots and bounces around.
- **Epochs** -- how many times the model sees the whole training set. More isn't always better
  (too many causes *overfitting* -- memorizing instead of learning).
- **Regularization** -- deliberate handicaps that stop the model from memorizing. Two kinds here:
  **dropout** (randomly ignore some neurons each step) and **weight decay** (an L2 penalty that
  keeps the weights small). Turn them up when the model overfits.
- **Activation function** -- the little nonlinearity after each layer that lets the network bend.
  `relu` is the modern default; `sigmoid`/`tanh` are the classics (and can stall a deep net).

First, let's *see* the learning rate matter: the cell below trains the same small CNN at three
learning rates and overlays the curves. **Change the numbers and re-run** -- this is your playground.

In [ ]:
# lets this cell run on its own -- e.g. if you restart the kernel and jump here
try:
    common, nbfig, device, tr224, va224
except NameError:
    import sys; sys.path.insert(0, ".")
    import common
    nbfig, device, tr224, va224 = common.playground_setup()

# ---- the control panel: change these and re-run ----
LEARNING_RATES = [3e-4, 1e-3, 3e-3]   # try adding 1e-2 (too big) or 1e-5 (too small)
EPOCHS = 8
# ----------------------------------------------------

runs = {}
for lr in LEARNING_RATES:
    print(f"training a fresh CNN at lr={lr} ...")
    model = common.make_small_cnn().to(device)
    runs[lr] = common.train_model(model, tr224, va224, epochs=EPOCHS, lr=lr, device=device, verbose=False)

# overlay: validation accuracy (left) and training loss (right) for each learning rate
fig, (a1, a2) = nbfig.fig(1, 2, figsize=(11, 4.2))
colors = nbfig.palette(len(LEARNING_RATES))
for c, (lr, hist) in zip(colors, runs.items()):
    ep = [h[0] for h in hist]
    a1.plot(ep, [h[1] for h in hist], "-o", color=c, label=f"lr={lr}")
    a2.plot(ep, [h[2] for h in hist], "-o", color=c, label=f"lr={lr}")
a1.set_title("validation accuracy"); a1.set_xlabel("epoch"); a1.set_ylabel("accuracy"); a1.legend()
a2.set_title("training loss"); a2.set_xlabel("epoch"); a2.set_ylabel("loss"); a2.legend()
nbfig.show(fig, "Same model, three learning rates")

Look at the shapes. The **middle** learning rate usually climbs fastest and settles highest. The
**smallest** one learns too slowly to get there in time. The **largest** one is jumpy, its loss
zig-zags because each step overshoots. That's the whole intuition behind tuning: find the step
size that's big enough to make progress but small enough to be stable. Try `1e-2` to watch it
break, then `1e-5` to watch it crawl.

### 3.4 The full dashboard: every dial in one place
Now the real playground. The control panel below exposes **all** the knobs at once, learning rate,
epochs, dropout, weight decay, and the activation function. Change any of them, re-run, and read
the curve. Some things to try:

- Set `DROPOUT = 0.0` and `WEIGHT_DECAY = 0.0` -- watch training loss fall fast but validation
  accuracy stall or wobble (that gap is overfitting).
- Push `DROPOUT = 0.5` or `WEIGHT_DECAY = 1e-3` -- training gets *harder* (loss falls slower) but
  the model often *generalizes* better.
- Swap `ACTIVATION = "sigmoid"` -- feel how a classic nonlinearity can slow a deeper network down.

In [ ]:
# lets this cell run on its own -- e.g. if you restart the kernel and jump here
try:
    common, nbfig, device, tr224, va224
except NameError:
    import sys; sys.path.insert(0, ".")
    import common
    nbfig, device, tr224, va224 = common.playground_setup()

# ================= THE CONTROL PANEL: change anything, re-run =================
LEARNING_RATE = 1e-3      # step size downhill
EPOCHS        = 10        # passes over the data
DROPOUT       = 0.3       # regularization: 0.0 = none, 0.5 = heavy
WEIGHT_DECAY  = 0.0       # L2 regularization: try 1e-4, 1e-3
ACTIVATION    = "relu"    # "relu", "leaky_relu", "gelu", "elu", "tanh", "sigmoid"
# =============================================================================

model = common.make_small_cnn(dropout=DROPOUT, activation=ACTIVATION).to(device)
history = common.train_model(model, tr224, va224, epochs=EPOCHS, lr=LEARNING_RATE,
                             weight_decay=WEIGHT_DECAY, device=device, verbose=False)

print(f"lr={LEARNING_RATE}  dropout={DROPOUT}  weight_decay={WEIGHT_DECAY}  activation={ACTIVATION}")
print(f"final validation accuracy: {history[-1][1]:.3f}")
nbfig.learning_curve(history, text="Your custom CNN").show()

Keep a little log as you go, `"dropout 0.3 -> 0.5: val_acc 0.71 -> 0.74"`, exactly like you'll do
in the Day 3 capstone. That habit, change one dial, measure, write it down, is the entire craft of
tuning a model. There's no secret best setting; there's only *measured* better.

## 4. ResNet50 -- transfer learning

Instead of learning vision from scratch, we take a 50-layer network already trained on a
million everyday photos (ImageNet) and **reuse its vision.** We freeze it and train only a
small new final layer for our yes/no question. This is the single biggest practical trick
in modern computer vision.

![Transfer learning: reuse the backbone, train a tiny head](img/transfer_learning.png)

In [ ]:
# TODO: build a pretrained ResNet50 with common.make_resnet50(pretrained=True)
# TODO: finetune for 3 epochs at lr=1e-3 on the 224px loaders
results["resnet"] = history[-1][1]
print(f"resnet val accuracy: {history[-1][1]:.3f}")

In [ ]:
nbfig.learning_curve(history, text="ResNet: high accuracy in just 3 epochs").show()

### 4.1 Where is it looking?
A real worry in medical AI: is the model looking at the *disease*, or at some artifact (a
bright edge, the camera label)? **Grad-CAM** highlights the pixels that most drove the
prediction. We want the heat on the retina, not the border.

In [ ]:
import numpy as np
from PIL import Image

imgs, _ = next(iter(va224))
cam, predicted = common.gradcam(resnet, imgs[0], device=device)
cam_up = np.array(Image.fromarray((cam * 255).astype("uint8")).resize((224, 224))) / 255.0

fig, (a1, a2) = nbfig.fig(1, 2, figsize=(8.5, 4.4))
base = common._denorm(imgs[0]).permute(1, 2, 0).numpy()
a1.imshow(base); a1.set_title("the eye", fontsize=11); a1.axis("off")
a2.imshow(base); a2.imshow(cam_up, cmap="inferno", alpha=0.5)
a2.set_title(f"where ResNet looked (pred: {common.GRADE_NAMES[predicted]})", fontsize=11)
a2.axis("off")
nbfig.show(fig, "Grad-CAM: the model's attention")

*Why did pretraining help so much with so little training?* The network already knew how to
see edges, textures, and shapes. We only had to teach it which combinations mean "referable."
That's the power of standing on a million photos' worth of prior learning.

## 5. Vision Transformer -- patches that pay attention

The newest rung. A **ViT** chops the image into patches, turns each patch into a vector, and
lets the patches *pay attention* to each other -- deciding which patches matter for the call.
It's the exact machinery behind the language models you've used, just patches instead of
words. (Much more on that tomorrow.)

![A ViT: image to patches to attention to prediction](img/vit_patches.png)

In [ ]:
# TODO: build a pretrained ViT with common.make_vit_base(pretrained=True)
# TODO: train the head for 5 epochs at lr=1e-3 on the 224px loaders
results["vit"] = history[-1][1]
print(f"vit val accuracy: {history[-1][1]:.3f}")

In [ ]:
nbfig.learning_curve(history, text="Vision Transformer").show()

On a few thousand images the ViT and the ResNet usually land close together -- transformers
are hungrier for data, so their real advantage shows at larger scale. The point isn't that
ViT always wins; it's that *attention* is a second, equally powerful way to see, and it's the
bridge to tomorrow.

## 6. The leaderboard you just built

Five models, same data, same yes/no question. Watch the climb -- and remember which idea
bought each step up.

In [ ]:
names = list(results.keys())
accs = [results[n] for n in names]

fig, ax = nbfig.fig(figsize=(7.5, 4))
bars = ax.bar(names, accs, color=nbfig.palette(len(names)))
for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width() / 2, a + 0.01, f"{a:.2f}", ha="center",
            fontweight="bold", family="DejaVu Sans Mono")
ax.set_ylabel("validation accuracy"); ax.set_ylim(0, 1)
nbfig.show(fig, "Same data, five models")

Each rung added exactly one idea: **non-linearity** (MLP), **spatial structure** (CNN),
**pretraining** (ResNet), **attention** (ViT). That is, in miniature, the last fifteen years
of computer vision.

## 7. Accuracy can lie -- and why that's dangerous here

Remember the class balance from section 0.3. In screening, the classes are lopsided and the
**costs are not symmetric**: telling a sick person they're fine (a *false negative*) is far
worse than a false alarm. A model can post a high accuracy while quietly missing the cases
that matter most. Always read the confusion matrix, not just the headline number.

In [ ]:
best = common.evaluate_classifier(lambda b: resnet(b).argmax(1), va224, device)
nbfig.confusion(best["y"], best["pred"], common.GRADE_NAMES,
                text="Best model: count the missed 'referable' eyes").show()
print("per-class recall:", {k: round(v, 2) for k, v in
      common.evaluate_classifier(lambda b: resnet(b).argmax(1), va224, device)["per_class"].items()})

The bottom-left cell -- truly *referable* eyes the model called *not referable* -- is the one
a clinician loses sleep over. This is why medical AI is judged on sensitivity and specificity,
not raw accuracy. We dig into exactly that vocabulary in the slides.

### 7.1 Play with it: slide the decision threshold
The model doesn't output "yes/no", it outputs a *probability*. **You** pick the cutoff for
action. Drag the slider and watch the trade-off happen live: lower the threshold to catch more
disease (sensitivity up) but trigger more false alarms (specificity down). There is no setting
that wins both, that's the whole lesson of the ROC curve, in your hands.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# self-heal: make sure nbfig exists even if you jumped here after a restart
try:
    nbfig
except NameError:
    import sys; sys.path.insert(0, "."); import common
    nbfig, device, tr224, va224 = common.playground_setup()

# use your trained model's real probabilities if they're in memory, else clear example scores
try:
    import torch
    resnet, device, va224
    resnet.eval(); _P, _Y = [], []
    with torch.no_grad():
        for xb, yb in va224:
            _P.append(torch.softmax(resnet(xb.to(device)), 1)[:, 1].cpu().numpy())
            _Y.append(yb.numpy())
    scores, truth = np.concatenate(_P), np.concatenate(_Y)
    source = "your ResNet's real predictions on the validation eyes"
except NameError:
    _rng = np.random.RandomState(0)
    truth = np.r_[np.zeros(180), np.ones(120)].astype(int)
    scores = np.r_[np.clip(_rng.beta(2, 5, 180), 0, 1), np.clip(_rng.beta(5, 2, 120), 0, 1)]
    source = "example scores (run the notebook top-to-bottom to use your real model)"

def threshold_demo(threshold=0.5):
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (truth == 1)).sum()); fn = int(((pred == 0) & (truth == 1)).sum())
    tn = int(((pred == 0) & (truth == 0)).sum()); fp = int(((pred == 1) & (truth == 0)).sum())
    sens, spec = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.hist(scores[truth == 0], bins=20, alpha=0.6, color=nbfig.TURQUOISE, label="not referable")
    ax.hist(scores[truth == 1], bins=20, alpha=0.6, color=nbfig.DEEPPINK, label="referable")
    ax.axvline(threshold, color=nbfig.INK, lw=2.5, ls="--")
    ax.set_xlabel("model's predicted probability of 'referable'"); ax.legend(loc="upper center")
    ax.set_title(f"threshold {threshold:.2f}   ->   sensitivity {sens:.0%} (missed {fn}),  "
                 f"specificity {spec:.0%} (false alarms {fp})", fontsize=9)
    plt.show()

print("scoring:", source)
try:
    from ipywidgets import interact, FloatSlider
    interact(threshold_demo, threshold=FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05))
except ImportError:
    print("ipywidgets not available -- showing a strict, balanced, and lenient cutoff:")
    for t in (0.75, 0.5, 0.25):
        threshold_demo(t)

## Bridge to Day 2

The Vision Transformer worked by splitting the image into patches and letting them attend to
each other:

```
   IMAGE                                     SENTENCE
   [patch][patch][patch]  --embed-->         "the"  "cat"  "sat"  --embed-->
        vectors                                    vectors
          |                                          |
       attention  (which patches matter?)         attention  (which words matter?)
          |                                          |
       prediction: referable?                     prediction: next word
```

That second column is a **Large Language Model**. Same machinery, different input. Tomorrow we
use one to read radiology reports and combine it with images. See you then.

## Stretch -- find a disagreement

If you finished early:

1. Find a validation eye where the **ResNet and the ViT predict differently**. Show the image
   and both predictions. What's unusual about it -- blurry, dark, an artifact?
2. Run Grad-CAM on a model's **wrong** prediction. Was it looking at the wrong thing?
3. Which model would you actually deploy in a clinic, and why? (Hint: it's not just accuracy.)